# Data Reading - csv

In [0]:
# Read CSV data using Spark's CSV data source.
# header=True → first row is treated as column names.
# inferSchema=True → Spark automatically infers column data types.
# In Databricks Free Edition, direct file/cloud-storage access is restricted,
# so file-based reading may not be available in the same way as a full Databricks setup.
'''
df = (
    spark.read
    .format('csv')
    .option('inferSchema', True)
    .option('header', True)
    .load('workspace.default.big_mart_sales')
)
'''

# Read an existing table registered in Unity Catalog as a Spark DataFrame.
# Format: catalog.schema.table
# In Databricks Free Edition, table-based access through Unity Catalog
# is commonly used for working with datasets.
df = spark.table('workspace.default.big_mart_sales')

# Key difference:
# spark.read  → reads data from a data source/file
# spark.table → reads an existing catalog table

In [0]:
df.display()

## Data Reading - JSON

In [0]:
df_json = spark.read.format('json')\
    .option('inferSchema',True)\
    .option('header',True)\
    .option('multiLine',False)\
    .load('/Volumes/workspace/default/sample_data/drivers.json')

In [0]:
df_json.display()

## Schema Definition

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 
                ''' 

In [0]:
df = spark.read.format('csv')\
    .schema(my_ddl_schema)\
    .option('header',True)\
    .load('/Volumes/workspace/default/sample_data/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

## structType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_Weight',StringType(),True),
    StructField('Item_Fat_Content',StringType(),True),
    StructField('Item_Visibility',StringType(),True),
    StructField('Item_MRP',StringType(),True),
    StructField('Outlet_Identifier',StringType(),True),
    StructField('Outlet_Establishment_Year',StringType(),True),
    StructField('Outlet_Size',StringType(),True),
    StructField('Outlet_Location_Type',StringType(),True),
    StructField('Outlet_Type',StringType(),True),
    StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df = spark.read.format('csv')\
    .schema(my_struct_schema)\
    .option('header',True)\
    .load('/Volumes/workspace/default/sample_data/BigMart Sales.csv')

In [0]:
df.printSchema()

# Transformations

## SELECT

In [0]:
df = (
    spark.read
    .format('csv')
    .option('inferSchema', True)
    .option('header', True)
    .load('workspace.default.big_mart_sales')
)

df = spark.table('workspace.default.big_mart_sales')

In [0]:
# this display the entire data frame that is in the memory
df.display()

In [0]:
# Just simply mentino the column names to get those columns
df.select("Item_Fat_Content","Item_Identifier","Item_Fat_Content").display()

In [0]:
# using col() function to extract columns
df.select(col("Item_Fat_Content"),col("Item_Identifier"),col("Item_Fat_Content")).display()

## ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

## Filter data

### Scenario 1

In [0]:
# filtering just based on one column
df.filter(col('Item_Fat_Content')=='Regular').display()

### Scenario 2

In [0]:
# applying filter condition on multiple columns
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).display()  

### Scenario 3

In [0]:
# using isNull and isin functions with columns to apply multiple and complex filter condition
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

## renaming existing column

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

## Creating a new column/modify existing column

### Scenario - 1

In [0]:
df_new = df.withColumn('flag',lit("new")) 

In [0]:
df_new.display()

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

### Scenario - 2

In [0]:
df_new_col = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

df_new_col.display()

## Type Casting

In [0]:
df_cast = df.withColumn('Item_Weight',col("Item_Weight").cast(StringType()))

In [0]:
df_cast.printSchema()

## Sorting/orderBy

### Scenario 1

In [0]:
df.sort(col('Item_Weight').desc()).display()

### Scenario 2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

### Scenario 3

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending = [0,0]).display()

### Scenario 4

In [0]:
df.sort(['Item_weight','Item_Visibility'], ascending = [0,1]).display()

## Limit

In [0]:
df.limit(10).display() 

## Drop

In [0]:
df.drop('Item_Visibility').display()

In [0]:
df.drop('Item_Visibility','Item_Type').display()

## Drop Duplicate records

### Scenario 1

In [0]:
df.dropDuplicates().display()

### Scenario 2

In [0]:
# dropping duplicates based on a column
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

## UNION and UNION BY NAME

### Preparing Dataframes

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)


In [0]:
df1.display()

In [0]:
df2.display()

### UNION

In [0]:
df1.union(df2).display()

### UNION BY NAME

In [0]:
data1 = [('kad','1',),
        ('sid','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.display()

In [0]:
df1.union(df2).display()

In [0]:
# union by name is useful when the columns in different dataframe are not in order
df1.unionByName(df2).display()

## String Functions

### initcap

In [0]:
df.select(initcap('Item_Type').alias('init_Item_Type')).display()

### lower

In [0]:
df.select(lower('Item_Type').alias('lower_Item_Type')).display()

### upper

In [0]:
df.select(upper('Item_Type').alias('upper_Item_Type')).display()

## Date Functions

### current_date

In [0]:
df = df.withColumn('curr_date',current_date())

df.display()

### date_add

In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))

df.display()

### date_sub

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

In [0]:
df = df.withColumn('week_before',date_add('curr_date',-7)) 

df.display()

### datediff

In [0]:
df = df.withColumn('datediff',datediff('week_after','curr_date'))

df.display()

### date_dateformat

In [0]:
df = df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))

df.display()

## Handling NULLs

### Dropping Nulls

In [0]:
# drops records which have nulls in all columns
df.dropna('all').display()

In [0]:

# If null value is present in any of the column of row, then it will drop that record.
df.dropna('any').display()

In [0]:
# Basically we test a record based on column. So if column is null, then it will drop that record.
df.dropna(subset=['Outlet_Size']).display()

In [0]:
df.display()

### Filling NULLs

In [0]:
# Fillinf all null values as 'NotAvailable'
df.fillna('NotAvailable').display()

In [0]:
# Fillinf all null values as 'NotAvailable' where column Outlet_Size is null
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

## SPLIT and Indexing

### SPLIT


In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

### Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

## Explode

In [0]:

df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))

In [0]:
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

In [0]:
df_exp.display()

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

## Group By

### Scenario-1

In [0]:
df.display()

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

### Scenario-2

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

### Scenario-3

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

### Scenario-4

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).display()

## Collect List

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

## PIVOT

In [0]:
df.select('Item_Type','Outlet_Size','Item_MRP').display()

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

## WHEN-OTHERWISE

In [0]:
df_cond = df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))

In [0]:
df_cond.display()

In [0]:
df_cond1 = df_cond.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                            .otherwise('Non_Veg'))

In [0]:
df_cond1.display()

## JOINS

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

### Inner Join

In [0]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner').display()

### Left Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

### Right Join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

### ANTI JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

## WINDOW Functions

### row_number()

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

### Rank vs Dense Rank

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

### Cumulative Sum

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

## User Defined Functions (UDF)

### Step 1

In [0]:
def my_func(x):
    return x*x

### Step 2

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

## Writing Data

### csv

In [0]:
df.write.format('csv')\
        .save('/FileStore/tables/CSV/data.csv')

### Append 

In [0]:
# This basically adds a new file with new data in the same directory
df.write.format('csv')\
        .mode('append')\
        .save('/FileStore/tables/CSV/data.csv')

In [0]:
# this will add new file to directory with new data in the path we have given
df.write.format('csv')\
        .mode('append')\
        .option('path','/FileStore/tables/CSV/data.csv')\
        .save()

### Overwrite

In [0]:
# This will overwrite the existing file
df.write.format('csv')\
.mode('overwrite')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

### Error

In [0]:
# This will throw an error
df.write.format('csv')\
.mode('error')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

### Ignore

In [0]:
# It will ignore the new data if the the file already exists in the path
df.write.format('csv')\
.mode('ignore')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

### Parquet file format

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

### Table

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')

## Spark SQL

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Type = 'Snack Foods';

In [0]:
# Saving result of your query into a dataframe
df_sql = spark.sql("select * from my_view where Item_Type = 'Snack Foods'")

In [0]:
df_sql.display()